# Make Baseline Predictions

Predict consumption using the previous hour and the same hour yesterday. Use the three-home example.

Open this notebook in `notebooks/M4/`. Run the cells in order. Saving again replaces this notebook's output files.


## Shared folder setup

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path("../..").resolve()
PROJECT_ROOT

## Choose the existing dataset

In [ ]:
from pathlib import Path
import pandas as pd
raw_path = PROJECT_ROOT / "data/raw/half_hour_readings_2013.csv"
raw_path

## Use M4's prototype

In [ ]:
homes = ["MAC000038", "MAC000045", "MAC001617"]
start = pd.Timestamp("2013-01-01", tz="UTC")
end = pd.Timestamp("2013-01-08", tz="UTC")
homes, start, end

## Read only the three homes into memory

In [ ]:
pieces = []
for chunk in pd.read_csv(raw_path, chunksize=200000, na_values=["Null", "null"]):
    pieces.append(chunk[chunk["LCLid"].isin(homes)])
data = pd.concat(pieces, ignore_index=True)
data.columns

## Parse the fields

In [ ]:
data = data.rename(columns={"energy(kWh/hh)": "kwh"})
data["tstp"] = pd.to_datetime(data["tstp"], utc=True)
data["kwh"] = pd.to_numeric(data["kwh"])
data.dtypes

## Keep exactly the first week's intervals

In [ ]:
data = data[(data["tstp"] > start) & (data["tstp"] <= end)].copy()
data.head()

## See the sample size per home

In [ ]:
data.groupby("LCLid").size()

## Derive the hourly start

In [ ]:
data["hour"] = (data["tstp"] - pd.Timedelta(minutes=30)).dt.floor("h")
data[["LCLid", "tstp", "hour", "kwh"]].head()

## Add the two half-hours

In [ ]:
grouped = data.groupby(["LCLid", "hour"])["kwh"].agg(["sum", "count"]).reset_index()
grouped.loc[grouped["count"] != 2, "sum"] = float("nan")
hourly_values = grouped.rename(columns={"sum": "kwh"})[["LCLid", "hour", "kwh"]]
grouped.head()

## Make every expected hour visible

In [ ]:
hours = pd.date_range(start, end, freq="h", inclusive="left")
grid = pd.MultiIndex.from_product([homes, hours], names=["LCLid", "hour"]).to_frame(index=False)
len(grid)

## Keep entirely absent hours as unknown

In [ ]:
hourly = grid.merge(hourly_values, on=["LCLid", "hour"], how="left")
hourly.head()

## Count missing hours

In [ ]:
hourly["kwh"].isna().groupby(hourly["LCLid"]).sum()

## Prepare the actual values

In [ ]:
lookup = hourly.set_index(["LCLid", "hour"])["kwh"]
predictions = hourly.rename(columns={"hour": "target_hour", "kwh": "actual_kwh"}).copy()
predictions.head()

## Predict using the previous hour

In [ ]:
previous_time = predictions["target_hour"] - pd.Timedelta(hours=1)
previous_keys = pd.MultiIndex.from_arrays([predictions["LCLid"], previous_time])
predictions["pred_prev_hour"] = lookup.reindex(previous_keys).to_numpy()
predictions.head()

## Predict using the same hour yesterday

In [ ]:
yesterday_time = predictions["target_hour"] - pd.Timedelta(hours=24)
yesterday_keys = pd.MultiIndex.from_arrays([predictions["LCLid"], yesterday_time])
predictions["pred_prev_day"] = lookup.reindex(yesterday_keys).to_numpy()
predictions.iloc[22:28]

## Check one row directly from raw data

In [ ]:
home = "MAC000038"
target = pd.Timestamp("2013-01-02 01:00", tz="UTC")
hand = pd.DataFrame({"field": ["actual_kwh", "pred_prev_hour", "pred_prev_day"], "offset": [0, 1, 24]})
hand["hour"] = target - pd.to_timedelta(hand["offset"], unit="h")
hand

## Look up the two ends for each hour

In [ ]:
raw_home = data[data["LCLid"] == home].set_index("tstp")["kwh"]
hand["first_end"] = hand["hour"] + pd.Timedelta(minutes=30)
hand["second_end"] = hand["hour"] + pd.Timedelta(hours=1)
hand["first_kwh"] = raw_home.reindex(hand["first_end"]).to_numpy()
hand["second_kwh"] = raw_home.reindex(hand["second_end"]).to_numpy()
hand["from_raw"] = hand["first_kwh"] + hand["second_kwh"]
hand

## Compare the hand calculation

In [ ]:
selected = predictions[(predictions["LCLid"] == home) & (predictions["target_hour"] == target)].iloc[0]
hand["from_predictions"] = selected[hand["field"]].to_numpy()
hand["matches"] = (hand["from_raw"] - hand["from_predictions"]).abs() < 1e-12
hand

## Inspect a gap

In [ ]:
gap_start = pd.Timestamp("2013-01-06 19:00", tz="UTC")
gap_rows = predictions[(predictions["LCLid"] == "MAC001617") & (predictions["target_hour"] >= gap_start)]
gap_rows.head(9)

## Set up M4's invented baseline checks

In [ ]:
test_hours = pd.date_range(start, periods=27, freq="h")
test = pd.DataFrame({"LCLid": "TEST", "hour": test_hours, "kwh": [i / 10 for i in range(27)]})
test.loc[25, "kwh"] = float("nan")
test = test.drop(index=3)
test.head()

## Calculate the invented previous-hour predictions

In [ ]:
test_lookup = test.set_index(["LCLid", "hour"])["kwh"]
test_predictions = test.rename(columns={"hour": "target_hour", "kwh": "actual_kwh"}).copy()
test_previous = test_predictions["target_hour"] - pd.Timedelta(hours=1)
test_keys = pd.MultiIndex.from_arrays([test_predictions["LCLid"], test_previous])
test_predictions["pred_prev_hour"] = test_lookup.reindex(test_keys).to_numpy()
test_predictions.head()

## Calculate the invented previous-day predictions

In [ ]:
test_yesterday = test_predictions["target_hour"] - pd.Timedelta(hours=24)
test_keys = pd.MultiIndex.from_arrays([test_predictions["LCLid"], test_yesterday])
test_predictions["pred_prev_day"] = test_lookup.reindex(test_keys).to_numpy()
test_results = test_predictions.set_index("target_hour")
test_results.tail(3)

## Check a normal hour and an absent hour

In [ ]:
test_results.loc[test_hours[[2, 4]], ["actual_kwh", "pred_prev_hour"]]

## Check the first hour and missing inputs

In [ ]:
test_results.loc[test_hours[[0, 24, 26]], ["pred_prev_hour", "pred_prev_day"]]

## Add the output labels

In [ ]:
predictions.insert(0, "task_id", "M4-W07-T01")
predictions["label"] = "PROTOTYPE"
predictions.head()

## Choose the output folder

In [ ]:
output_dir = PROJECT_ROOT / 'results'
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

## Save the predictions

In [ ]:
prediction_path = output_dir / "baseline_predictions_sample.csv"
predictions.to_csv(prediction_path, index=False)
prediction_path

## Save the hand-check evidence

In [ ]:
hand_path = output_dir / "baseline_check_sample.csv"
hand.to_csv(hand_path, index=False)
hand_path